# Overview

The workflow is:

1. Read the step data (`data/pig_all_steps_*.csv`), which is in **EPSG:3112**
   (GDA94 / Geoscience Australia Lambert).
2. Take the bounding box of all step start *and* end coordinates, buffer it by
   10 km, and turn it into an `ee.Geometry` (the AOI).
3. Take the temporal range of the tracking data (`t1_` to `t2_`) and build a
   **median cloud-free Sentinel-2 composite** over that window, using the
   Cloud Score+ mask.
4. Derive **NDVI** and **NDWI** from the same composite.
5. Grab the **AlphaEarth / Satellite Embedding** annual image (64 bands) for a
   chosen year.
6. Export each product, either locally to `mapping/` (small rasters) or to the
   Google Drive folder `feral_pig_mapping` (anything bigger).

Everything you are likely to want to change lives in the *Configuration* cell.

# Load packages

In [5]:
import io
import zipfile
from datetime import timedelta
from pathlib import Path

import ee
import geemap
import pandas as pd
import requests
from pyproj import Transformer

# Initialise GEE

You will need to authenticate your Google account the first time you run this.
`ee.Authenticate()` opens a browser window; the resulting token is cached in
`~/.config/earthengine/` so subsequent runs skip straight to `ee.Initialize()`.

In [6]:
try:
    ee.Initialize()
    print("Earth Engine initialised")
except Exception:
    ee.Authenticate()
    ee.Initialize()
    print("Earth Engine authenticated and initialised")

Earth Engine initialised


# Configuration

One cell to rule them all - change these, then run the rest of the notebook top
to bottom.

In [16]:
# --- Tracking data ---------------------------------------------------------
CSV_PATH  = 'data/pig_all_steps_20260707.csv'
DATA_CRS  = 'EPSG:3112'          # GDA94 / GA Lambert - the CRS of x1_, y1_, ...
X_COLS    = ['x1_', 'x2_']       # all columns holding easting  values
Y_COLS    = ['y1_', 'y2_']       # all columns holding northing values
TIME_COLS = ['t1_', 't2_']       # all columns holding timestamps

# --- Area of interest ------------------------------------------------------
AOI_NAME = 'feral_pig_aoi'       # used as the filename prefix for every export
BUFFER_M = 2500                  # buffer around the data bounding box

# --- Sentinel-2 ------------------------------------------------------------
# Which bands to export. The composite is always built with all optical bands
# (so the indices can be computed), this only controls what gets written out.
S2_BAND_MODE    = 'full'         # 'rgb' | 'nir' | 'full'
CLEAR_THRESHOLD = 0.65           # Cloud Score+ 'cs_cdf': 0 = cloud, 1 = clear
MAX_CLOUDY_PCT  = 90             # drop granules that are almost entirely cloud
PAD_DAYS        = 0              # widen the tracking window by n days each end

# --- AlphaEarth (Satellite Embedding V1, annual, 64 bands, 10 m) -----------
# None -> automatically pick the available year that best matches the tracking
# period. Set an integer (e.g. 2024) to force a specific year.
ALPHAEARTH_YEAR = None

# --- Export ----------------------------------------------------------------
EXPORT_SCALE   = 20              # metres per pixel (S2 and AlphaEarth are 10 m)
EXPORT_CRS     = 'EPSG:3112'     # match the tracking data so rasters line up
DESTINATION    = 'auto'          # 'auto' | 'local' | 'drive'
LOCAL_DIR      = 'mapping'       # where local GeoTIFFs are written
DRIVE_FOLDER   = 'feral_pig_mapping'   # Google Drive folder for big exports
LOCAL_MAX_MB   = 32              # 'auto' sends anything larger to Drive
                                 # (getDownloadURL caps out around 32-48 MB)
SCALE_TO_INT16 = True            # store reflectance/indices as int16 (x10000)
                                 # instead of float32 - halves the file size

# Build the AOI from the tracking data

The step data stores a start (`x1_`, `y1_`) and an end (`x2_`, `y2_`) coordinate
per row, so the bounding box is taken over both. Coordinates are projected
(metres), which makes the 10 km buffer straightforward - but Earth Engine wants
geographic coordinates, so we reproject the buffered box corners to WGS84.

The box edges are densified before reprojecting: a straight line in EPSG:3112 is
a curve in lon/lat, so sampling along each edge and taking the min/max guarantees
the WGS84 rectangle still contains the whole buffered box.

In [12]:
# Read only the columns we need - the file is ~250k rows.
tracks = pd.read_csv(CSV_PATH, usecols=X_COLS + Y_COLS + TIME_COLS + ['id'])

# --- Spatial extent (projected CRS, metres) --------------------------------
xmin = tracks[X_COLS].min().min() - BUFFER_M
xmax = tracks[X_COLS].max().max() + BUFFER_M
ymin = tracks[Y_COLS].min().min() - BUFFER_M
ymax = tracks[Y_COLS].max().max() + BUFFER_M

print(f"Tracking data:   {len(tracks):,} steps from {tracks['id'].nunique()} individuals")
print(f"Buffered extent: {(xmax - xmin) / 1000:.1f} km x {(ymax - ymin) / 1000:.1f} km "
      f"({DATA_CRS}, {BUFFER_M / 1000:.0f} km buffer)")

# --- Reproject the buffered box to WGS84 -----------------------------------
to_wgs84 = Transformer.from_crs(DATA_CRS, 'EPSG:4326', always_xy=True)

# Densify: 50 points along each edge of the rectangle rather than just 4 corners.
n_edge = 50
edge_x = [xmin + i * (xmax - xmin) / n_edge for i in range(n_edge + 1)]
edge_y = [ymin + i * (ymax - ymin) / n_edge for i in range(n_edge + 1)]
box_x = edge_x + edge_x + [xmin] * (n_edge + 1) + [xmax] * (n_edge + 1)
box_y = [ymin] * (n_edge + 1) + [ymax] * (n_edge + 1) + edge_y + edge_y

lons, lats = to_wgs84.transform(box_x, box_y)

# The AOI: a non-geodesic rectangle (edges follow lines of constant lon/lat),
# which is what you want for a bounding box used to clip/export imagery.
aoi = ee.Geometry.Rectangle(
    [min(lons), min(lats), max(lons), max(lats)], proj='EPSG:4326', geodesic=False
)

print(f"AOI (WGS84):     {min(lons):.4f}, {min(lats):.4f} -> {max(lons):.4f}, {max(lats):.4f}")
print(f"AOI area:        {aoi.area(maxError=1).getInfo() / 1e6:,.0f} km2")

Tracking data:   246,323 steps from 28 individuals
Buffered extent: 70.2 km x 51.5 km (EPSG:3112, 2 km buffer)
AOI (WGS84):     132.1250, -12.5592 -> 132.7633, -12.0950
AOI area:        3,579 km2


## Temporal extent

The Sentinel-2 composite covers the full span of the tracking data. `PAD_DAYS`
lets you widen that window (useful if the tracking period is short and the wet
season leaves you with too few clear observations).

In [13]:
# Parse every timestamp column and take the overall min/max.
times = pd.concat([pd.to_datetime(tracks[c], utc=True) for c in TIME_COLS])
track_start, track_end = times.min(), times.max()

start_date = (track_start - timedelta(days=PAD_DAYS)).strftime('%Y-%m-%d')
# +1 day because Earth Engine's filterDate() end is exclusive.
end_date = (track_end + timedelta(days=PAD_DAYS + 1)).strftime('%Y-%m-%d')

print(f"Tracking period:  {track_start:%Y-%m-%d} to {track_end:%Y-%m-%d}")
print(f"Imagery window:   {start_date} to {end_date} (end exclusive)")

Tracking period:  2025-09-12 to 2026-06-22
Imagery window:   2025-09-12 to 2026-06-23 (end exclusive)


## Preview the AOI

A quick sanity check that the AOI covers the tracking data. The GPS fixes are
subsampled before being sent to Earth Engine - drawing a quarter of a million
points on a web map is neither fast nor informative.

In [14]:
# Subsample the step start points down to a manageable number for plotting.
max_points = 2000
step = max(1, len(tracks) // max_points)
pts = tracks.iloc[::step]
pt_lons, pt_lats = to_wgs84.transform(pts['x1_'].values, pts['y1_'].values)
fixes = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([float(lon), float(lat)]))
    for lon, lat in zip(pt_lons, pt_lats)
])

Map = geemap.Map()
Map.add_basemap('SATELLITE')
Map.addLayer(fixes, {'color': 'yellow'}, f'GPS fixes (every {step}th)')
Map.addLayer(aoi, {'color': 'red'}, 'AOI (bbox + 10 km)')
Map.centerObject(aoi)

# Displaying `Map` works in VS Code / Jupyter. When rendering with Quarto the
# widget sometimes gets dropped, in which case write it out and embed instead:
# Map.to_html(filename='satellite_map_aoi.html'); HTML('satellite_map_aoi.html')
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

# Sentinel-2 median composite

Cloud masking uses [Cloud Score+](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_CLOUD_SCORE_PLUS_V1_S2_HARMONIZED),
which is linked onto the S2 collection with `linkCollection()` (the two
collections share image IDs, so this is a straight join). Compared with the
QA60 bitmask it handles thin cloud and cloud shadow far better.

`cs_cdf` is the CDF-based quality score, generally more stable for time series
than the raw `cs` band:

- 0.60 - permissive: more observations kept, some haze survives
- 0.65 - balanced (Google's suggested default, used here)
- 0.80+ - strict: cleanest composites, but risks holes in cloudy regions

In [17]:
# Sentinel-2 optical band sets. The composite always carries ALL of these so
# that NDVI/NDWI can be computed; S2_BAND_MODE only controls what is exported.
S2_ALL_BANDS = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7',
                'B8', 'B8A', 'B9', 'B11', 'B12']

S2_BAND_SETS = {
    'rgb':  ['B4', 'B3', 'B2'],                  # true colour
    'nir':  ['B8', 'B4', 'B3', 'B2'],            # false colour + NIR
    'full': S2_ALL_BANDS,                        # everything (excl. B10)
}


def mask_s2_clouds(image):
    """Mask an S2 image with Cloud Score+ and rescale reflectance to 0-1.

    Assumes the 'cs' / 'cs_cdf' bands have already been linked onto the image.
    """
    mask = image.select('cs_cdf').gte(CLEAR_THRESHOLD)
    return (image.divide(10000)          # DN -> surface reflectance (0-1)
                 .updateMask(mask)       # drop everything below the threshold
                 .copyProperties(image, ['system:time_start']))


def build_s2_median(start_date, end_date, aoi):
    """Median of all cloud-free Sentinel-2 pixels in the window, over the AOI."""
    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(aoi)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUDY_PCT)))

    cs_plus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
    s2_linked = s2.linkCollection(cs_plus, ['cs', 'cs_cdf'])

    print(f"Found {s2.size().getInfo()} S2 granules for {start_date} to {end_date}")

    # Per-pixel median over the cloud-free observations: outliers (haze,
    # unmasked cloud edges, water glint) get squeezed out far better than
    # they would by a mean.
    composite = s2_linked.map(mask_s2_clouds).median().select(S2_ALL_BANDS)

    return composite.clip(aoi).set({
        'start_date': start_date,
        'end_date': end_date,
        'clear_threshold': CLEAR_THRESHOLD,
    })

In [18]:
s2_median = build_s2_median(start_date, end_date, aoi)
print("Composite bands:", s2_median.bandNames().getInfo())

Found 94 S2 granules for 2025-09-12 to 2026-06-23
Composite bands: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']


# Spectral indices: NDVI and NDWI

Both are normalised differences, so they are unaffected by the 0-1 rescaling
above and land in the range [-1, 1]:

- **NDVI** = (NIR - Red) / (NIR + Red) = (B8 - B4) / (B8 + B4) - green
  vegetation vigour/cover.
- **NDWI** = (Green - NIR) / (Green + NIR) = (B3 - B8) / (B3 + B8) - McFeeters'
  index, which highlights *open water* (positive over water, negative over land).
- **NDMI** = (NIR - SWIR) / (NIR + SWIR) = (B8 - B11) / (B8 + B11) - Gao's
  index, sometimes also called NDWI. This is vegetation/soil *moisture* rather
  than open water, which is arguably the more useful covariate for pig habitat,
  so it is included as a bonus band.

Note these are indices *of the median composite*, not the median of per-scene
indices. For a seasonal-average covariate the difference is small, and this way
NDVI/NDWI are guaranteed consistent with the exported reflectance.

In [19]:
def add_indices(composite):
    """NDVI, NDWI (McFeeters) and NDMI (Gao) as a single 3-band image."""
    ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = composite.normalizedDifference(['B3', 'B8']).rename('NDWI')
    ndmi = composite.normalizedDifference(['B8', 'B11']).rename('NDMI')
    return ee.Image.cat([ndvi, ndwi, ndmi]).clip(aoi)


s2_indices = add_indices(s2_median)
print("Index bands:", s2_indices.bandNames().getInfo())

Index bands: ['NDVI', 'NDWI', 'NDMI']


# AlphaEarth (Satellite Embedding V1)

[`GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL`](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_SATELLITE_EMBEDDING_V1_ANNUAL)
is a 64-band, 10 m, **annual** embedding: each pixel is a unit-length vector
(band values in [-1, 1]) summarising a full year of optical, radar and
elevation data. Because it is annual, you have to pick a year - hence the
`ALPHAEARTH_YEAR` option.

Each annual image is stamped 1 January of the year it *covers* (the 2024 image
summarises Jan-Dec 2024). If `ALPHAEARTH_YEAR` is `None` we pick the available
year closest to the midpoint of the tracking period.

In [20]:
ALPHAEARTH_ID = 'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'

# Which years does the collection actually cover over our AOI?
ae_collection = ee.ImageCollection(ALPHAEARTH_ID).filterBounds(aoi)
available_years = sorted({
    pd.to_datetime(t, unit='ms').year
    for t in ae_collection.aggregate_array('system:time_start').getInfo()
})
print("AlphaEarth years available for this AOI:", available_years)


def pick_alphaearth_year(requested, available, track_start, track_end):
    """Resolve ALPHAEARTH_YEAR, defaulting to the year nearest the tracking midpoint."""
    if requested is not None:
        if requested not in available:
            raise ValueError(f"{requested} not available - choose from {available}")
        return requested

    # Midpoint of the tracking period, snapped to the nearest available year.
    midpoint_year = (track_start + (track_end - track_start) / 2).year
    year = min(available, key=lambda y: abs(y - midpoint_year))
    if year != midpoint_year:
        print(f"Tracking midpoint is {midpoint_year}, which has no embedding yet - "
              f"falling back to {year}")
    return year


alphaearth_year = pick_alphaearth_year(
    ALPHAEARTH_YEAR, available_years, track_start, track_end
)

# One image per year per tile, so mosaic the tiles covering the AOI.
alphaearth = (ae_collection
              .filterDate(f'{alphaearth_year}-01-01', f'{alphaearth_year + 1}-01-01')
              .mosaic()
              .clip(aoi))

print(f"Using AlphaEarth year {alphaearth_year} "
      f"({alphaearth.bandNames().size().getInfo()} bands)")

AlphaEarth years available for this AOI: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Tracking midpoint is 2026, which has no embedding yet - falling back to 2025
Using AlphaEarth year 2025 (64 bands)


# Exporting

Two routes out of Earth Engine:

- **Local** (`image.getDownloadURL`) - synchronous, the file lands in
  `mapping/` immediately, but Earth Engine caps a single request at roughly
  32-48 MB, so this only works for small rasters (small AOI, few bands, or a
  coarse `EXPORT_SCALE`).
- **Drive** (`ee.batch.Export.image.toDrive`) - asynchronous batch task, no
  practical size limit. Very large rasters are automatically split into
  multiple numbered GeoTIFFs.

`DESTINATION = 'auto'` estimates the output size and picks for you. Be warned:
at 10 m over this AOI, the 64-band AlphaEarth image is on the order of
10 GB - Drive is the only realistic option, and increasing `EXPORT_SCALE` to
20 or 30 m is worth considering if you don't need full resolution.

In [21]:
# Started Drive tasks accumulate here so they can be monitored later.
TASKS = []


def estimate_size_mb(image, region, scale, bytes_per_pixel):
    """Rough uncompressed size of an export, in MB."""
    n_bands = image.bandNames().size().getInfo()
    n_pixels = region.area(maxError=1).getInfo() / (scale ** 2)
    return n_pixels * n_bands * bytes_per_pixel / 1e6


def to_int16(image, factor=10_000):
    """Pack float bands into int16 by scaling (undo with / factor on read)."""
    return image.multiply(factor).round().toInt16()


def export_image(image, name, region=None, scale=None, destination=None,
                 int16=False):
    """Export an image locally or to Drive, choosing automatically if asked.

    int16=True scales the image by 10,000 and casts to int16 before export.
    Note that pixels which remained masked (e.g. permanent cloud) are written
    as 0 - unmask() them to a sentinel value first if you need to tell the two
    apart.
    """
    region = aoi if region is None else region
    scale = EXPORT_SCALE if scale is None else scale
    destination = (DESTINATION if destination is None else destination).lower()

    if int16:
        image = to_int16(image)

    bytes_per_pixel = 2 if int16 else 4
    size_mb = estimate_size_mb(image, region, scale, bytes_per_pixel)
    print(f"\n{name}: ~{size_mb:,.0f} MB uncompressed at {scale} m "
          f"({'int16' if int16 else 'float32'})")

    if destination == 'auto':
        destination = 'local' if size_mb <= LOCAL_MAX_MB else 'drive'
        print(f"  -> auto-selected '{destination}' (local limit {LOCAL_MAX_MB} MB)")

    if destination == 'local':
        out_dir = Path(LOCAL_DIR)
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"{name}.tif"

        url = image.getDownloadURL({
            'scale': scale,
            'crs': EXPORT_CRS,
            'region': region,
            'format': 'GEO_TIFF',
            'filePerBand': False,      # one multi-band GeoTIFF, not one per band
        })
        response = requests.get(url, timeout=600)
        response.raise_for_status()

        # Earth Engine sometimes hands back a zipped GeoTIFF rather than a bare
        # one ('PK' is the zip magic number), so handle both.
        if response.content[:2] == b'PK':
            with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
                tif_name = next(n for n in zf.namelist() if n.lower().endswith('.tif'))
                out_path.write_bytes(zf.read(tif_name))
        else:
            out_path.write_bytes(response.content)

        print(f"  -> saved {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)")
        return out_path

    if destination == 'drive':
        task = ee.batch.Export.image.toDrive(
            image=image,
            description=name[:100],        # EE caps description length
            folder=DRIVE_FOLDER,
            fileNamePrefix=name,
            region=region,
            scale=scale,
            crs=EXPORT_CRS,
            maxPixels=1e13,
            fileFormat='GeoTIFF',
            formatOptions={'cloudOptimized': True},
        )
        task.start()
        TASKS.append(task)
        print(f"  -> started Drive export to '{DRIVE_FOLDER}/{name}' (task {task.id})")
        return task

    raise ValueError("destination must be 'auto', 'local' or 'drive'")

# Run the exports

File names carry the AOI name, the product, and the period/year so that
different runs don't overwrite each other.

## Sentinel-2 median composite

In [22]:
period_tag = f"{start_date}_to_{end_date}"
s2_export = s2_median.select(S2_BAND_SETS[S2_BAND_MODE])

export_image(
    s2_export,
    name=f"{AOI_NAME}_S2_median_{S2_BAND_MODE}_{period_tag}",
    int16=SCALE_TO_INT16,   # reflectance x10000, i.e. back to the original DNs
)


feral_pig_aoi_S2_median_full_2025-09-12_to_2026-06-23: ~215 MB uncompressed at 20 m (int16)
  -> auto-selected 'drive' (local limit 32 MB)
  -> started Drive export to 'feral_pig_mapping/feral_pig_aoi_S2_median_full_2025-09-12_to_2026-06-23' (task ZOBI4BSG4CQLALUK6E7M3IFH)


<Task ZOBI4BSG4CQLALUK6E7M3IFH EXPORT_IMAGE: feral_pig_aoi_S2_median_full_2025-09-12_to_2026-06-23 (UNSUBMITTED)>

## NDVI / NDWI / NDMI

In [23]:
export_image(
    s2_indices,
    name=f"{AOI_NAME}_S2_indices_{period_tag}",
    int16=SCALE_TO_INT16,   # indices x10000, so divide by 10000 on read
)


feral_pig_aoi_S2_indices_2025-09-12_to_2026-06-23: ~54 MB uncompressed at 20 m (int16)
  -> auto-selected 'drive' (local limit 32 MB)
  -> started Drive export to 'feral_pig_mapping/feral_pig_aoi_S2_indices_2025-09-12_to_2026-06-23' (task O4FOCJFZI3M5J2HAC6V5OHZ7)


<Task O4FOCJFZI3M5J2HAC6V5OHZ7 EXPORT_IMAGE: feral_pig_aoi_S2_indices_2025-09-12_to_2026-06-23 (UNSUBMITTED)>

## AlphaEarth embedding

Left as float32: the embedding values are already unit-scaled and the whole
point of them is fine-grained numerical structure, so packing them into int16
is a false economy. To trim the size, coarsen `EXPORT_SCALE` or select a subset
of the 64 bands instead.

In [24]:
export_image(
    alphaearth,
    name=f"{AOI_NAME}_alphaearth_{alphaearth_year}",
    int16=False,
)


feral_pig_aoi_alphaearth_2025: ~2,291 MB uncompressed at 20 m (float32)
  -> auto-selected 'drive' (local limit 32 MB)
  -> started Drive export to 'feral_pig_mapping/feral_pig_aoi_alphaearth_2025' (task JWDK554ELLL5ELEOSO6QKFXC)


<Task JWDK554ELLL5ELEOSO6QKFXC EXPORT_IMAGE: feral_pig_aoi_alphaearth_2025 (UNSUBMITTED)>

## Monitor the Drive tasks

Drive exports run asynchronously on Google's servers - this notebook can be
closed and the tasks will keep going. Re-run the cell below to check progress,
or watch them at [code.earthengine.google.com](https://code.earthengine.google.com/tasks).
Finished files appear in Google Drive under `feral_pig_mapping`.

In [25]:
def print_task_status(tasks=None):
    """Print the current state of each export task started in this session."""
    tasks = TASKS if tasks is None else tasks
    if not tasks:
        print("No Drive tasks started in this session.")
        return
    for task in tasks:
        status = task.status()
        line = f"{status['description']:<55} {status['state']:<12}"
        if status.get('error_message'):
            line += f" {status['error_message']}"
        print(line)


print_task_status()

feral_pig_aoi_S2_median_full_2025-09-12_to_2026-06-23   RUNNING     
feral_pig_aoi_S2_indices_2025-09-12_to_2026-06-23       RUNNING     
feral_pig_aoi_alphaearth_2025                           RUNNING     


# Visual check

Preview the products in Earth Engine before waiting on (or trusting) the
exports. The layers below are the un-exported, un-rescaled versions, so
reflectance is 0-1 and the indices are -1 to 1.

In [26]:
Map2 = geemap.Map()
Map2.add_basemap('SATELLITE')

# True colour composite: 0-0.3 reflectance covers most land surfaces.
Map2.addLayer(s2_median, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3},
              'S2 median (true colour)')
Map2.addLayer(s2_indices.select('NDVI'),
              {'min': -0.2, 'max': 0.9, 'palette': ['brown', 'white', 'darkgreen']},
              'NDVI')
Map2.addLayer(s2_indices.select('NDWI'),
              {'min': -0.5, 'max': 0.5, 'palette': ['saddlebrown', 'white', 'navy']},
              'NDWI', shown=False)

# There is no meaningful "true colour" for a 64-dimensional embedding - mapping
# three arbitrary axes to RGB just makes structurally similar areas look similar.
Map2.addLayer(alphaearth, {'bands': ['A01', 'A16', 'A09'], 'min': -0.3, 'max': 0.3},
              f'AlphaEarth {alphaearth_year} (bands A01/A16/A09)', shown=False)

Map2.addLayer(aoi, {'color': 'red'}, 'AOI')
Map2.centerObject(aoi)
Map2

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…